In [4]:
"""
STEP 1 - Generate two completely independent synthetic glacial-lake datasets
===========================================================================

Dataset A  (TRAINING)  -> data/train_lakes.csv
    "Synthetic Range A": a monsoon-influenced, debris-rich, Himalaya-like range
    (27.5-31.0 N, 80.5-90.0 E). Used ONLY for model training and conformal
    calibration in Step 2.

Dataset B  (ANALYSIS)  -> data/analysis_features.csv   (what the model sees)
                          data/analysis_answer_key.csv (hidden truth)
    "Synthetic Range B": a dry, continental, Pamir/Tien-Shan-like range
    (38.0-41.5 N, 70.0-79.5 E). Generated by a separate generator with its own
    random stream, its own geography, its own valleys, its own climate field
    and its own parameter distributions (more ice-dammed lakes, cleaner and
    steeper glaciers, a less negative mass balance, faster warming, wetter
    trend). Step 3 predicts it WITHOUT retraining. The answer key is read only
    at the very end of Step 3 to check how well the pipeline did.

What the two datasets share
---------------------------
They share ONE thing: the laws of physics (`lake_response`), i.e. how a lake's
area responds to melt, glacier contact, debris, warming and so on. That is
deliberate and unavoidable: if the physics differed between the datasets there
would be nothing a model could learn that transfers. They share no lakes, no
coordinates, no valleys, no random numbers and no parameter distributions.

Target
------
y = log(A_2020 / A_2000), the log area ratio over two inventory epochs.
Positive = growing lake, negative = shrinking/drained lake.

Run:  python step1_generate_data.py            (default seeds)
      python step1_generate_data.py --analysis-seed random
In a Jupyter / ArcGIS Pro notebook:
      main()                                   (default seeds)
      main(["--analysis-seed", "random"])
"""
from __future__ import annotations

import argparse
import json
import os

import numpy as np
import pandas as pd

DATA_DIR = "data"
FIG_DIR = "figures"

# Columns the model may use (lat/lon/valley are for maps and blocking only).
FEATURES = [
    "elevation_m", "area_2000_km2", "glacier_area_km2", "dist_to_glacier_km",
    "glacier_slope_deg", "debris_cover_frac", "mass_balance_mwe",
    "temp_trend_c_dec", "precip_trend_pct_dec", "surround_slope_deg",
    "dam_face_slope_deg", "dam_type",
]


# ---------------------------------------------------------------------------
# 1. Shared physics ("nature"). Only Step 1 knows this function.
# ---------------------------------------------------------------------------
def lake_response(df: pd.DataFrame, rng: np.random.Generator,
                  spatial_noise: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (y, drained_flag): 20-year log area change for each lake.

    Physically inspired terms
      * proglacial expansion: lakes touching a thinning glacier grow by calving
        and terminus retreat; stronger with debris cover (stagnant tongues)
      * meltwater supply from the contributing glacier area
      * warming accelerates growth of glacier-fed lakes
      * precipitation trend adds/removes water for all lakes
      * supraglacial ponds coalesce on low-slope, debris-covered tongues
      * small lakes change proportionally more than large ones
      * bedrock basins cannot expand much; ice dams are unstable
      * rare outburst/drainage events cause abrupt shrinkage
    """
    melt = np.clip(-df["mass_balance_mwe"].to_numpy(), 0.0, None)
    contact = np.exp(-df["dist_to_glacier_km"].to_numpy() / 0.8)
    debris = df["debris_cover_frac"].to_numpy()
    slope = df["glacier_slope_deg"].to_numpy()
    garea = df["glacier_area_km2"].to_numpy()
    a1 = df["area_2000_km2"].to_numpy()
    temp = df["temp_trend_c_dec"].to_numpy()
    prec = df["precip_trend_pct_dec"].to_numpy()
    elev = df["elevation_m"].to_numpy()
    dam = df["dam_type"].to_numpy()

    low_slope = 1.0 / (1.0 + np.exp((slope - 5.0) / 1.2))
    dam_mult = np.select([dam == "bedrock", dam == "ice"], [0.45, 0.85], 1.0)

    mu = (
        1.10 * contact * melt * (1.0 + 0.8 * debris)
        + 0.45 * np.tanh(garea / 12.0) * melt
        + 0.90 * temp * contact
        + 0.35 * low_slope * debris * contact
        - 0.06 * (np.log(a1) + 2.5)
        - 0.10 * np.tanh((elev - 4600.0) / 500.0) * (1 - contact)
    ) * dam_mult + 0.012 * prec - 0.05

    # Heteroscedastic noise: contact lakes and small lakes are less predictable,
    # ice-dammed lakes are the least predictable.
    sigma = (0.020 + 0.060 * contact + 0.015 / np.sqrt(a1 / 0.05 + 0.2)
             + np.where(dam == "ice", 0.060, 0.0))
    y = mu + sigma * rng.standard_normal(len(df)) + spatial_noise

    # Partial or full drainage (outburst) events: rare, mostly ice/moraine dams.
    p_drain = (0.002 + np.where(dam == "ice", 0.03, 0.0)
               + np.where(dam == "moraine", 0.008 * contact, 0.0))
    drained = rng.random(len(df)) < p_drain
    y = y - drained * rng.uniform(0.25, 0.8, len(df))
    return y, drained


# ---------------------------------------------------------------------------
# 2. Helpers for geography
# ---------------------------------------------------------------------------
def smooth_field(rng, lat, lon, n_waves=12, length_deg=2.0):
    """Spatially smooth random field (random Fourier features), unit variance."""
    w = rng.normal(0, 1.0 / length_deg, size=(n_waves, 2))
    b = rng.uniform(0, 2 * np.pi, n_waves)
    f = np.cos(np.outer(lat, w[:, 0]) + np.outer(lon, w[:, 1]) + b).sum(1)
    return (f - f.mean()) / (f.std() + 1e-9)


def place_lakes(rng, n_lakes, n_valleys, lat_rng, lon_rng, spread_deg):
    centers = np.column_stack([rng.uniform(*lat_rng, n_valleys),
                               rng.uniform(*lon_rng, n_valleys)])
    size = rng.dirichlet(np.full(n_valleys, 1.5))          # unequal valley sizes
    valley = rng.choice(n_valleys, n_lakes, p=size)
    lat = np.clip(centers[valley, 0] + rng.normal(0, spread_deg, n_lakes), *lat_rng)
    lon = np.clip(centers[valley, 1] + rng.normal(0, spread_deg * 1.3, n_lakes), *lon_rng)
    return lat, lon, valley


def _lognormal(rng, median, sigma, n):
    return np.exp(np.log(median) + sigma * rng.standard_normal(n))


# ---------------------------------------------------------------------------
# 3. Generator A - training range
# ---------------------------------------------------------------------------
def generate_training_range(seed: int, n_lakes: int = 5000) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    lat, lon, valley = place_lakes(rng, n_lakes, 60, (27.5, 31.0), (80.5, 90.0), 0.12)
    nv = valley.max() + 1
    v_elev = rng.normal(4950, 260, nv)        # valley-level shared traits
    v_glac = _lognormal(rng, 9.0, 0.6, nv)
    f_temp = smooth_field(rng, lat, lon, length_deg=2.5)
    f_mb = smooth_field(rng, lat, lon, length_deg=1.8)
    f_prec = smooth_field(rng, lat, lon, length_deg=3.0)

    df = pd.DataFrame({
        "lake_id": [f"A{i:05d}" for i in range(n_lakes)],
        "lat": lat, "lon": lon, "valley": [f"A-V{v:02d}" for v in valley],
        "elevation_m": v_elev[valley] + rng.normal(0, 140, n_lakes),
        "area_2000_km2": np.clip(_lognormal(rng, 0.08, 1.0, n_lakes), 0.004, 6.0),
        "glacier_area_km2": np.clip(v_glac[valley] * _lognormal(rng, 1.0, 0.7, n_lakes), 0.1, 120),
        "dist_to_glacier_km": np.clip(rng.exponential(1.2, n_lakes), 0.0, 8.0),
        "glacier_slope_deg": np.clip(rng.gamma(4.0, 1.5, n_lakes), 0.5, 30),
        "debris_cover_frac": rng.beta(2.2, 3.0, n_lakes),
        "mass_balance_mwe": -0.45 + 0.12 * f_mb + rng.normal(0, 0.08, n_lakes),
        "temp_trend_c_dec": 0.30 + 0.06 * f_temp + 0.015 * (lat - 29) + rng.normal(0, 0.03, n_lakes),
        "precip_trend_pct_dec": -2.0 + 3.0 * f_prec + rng.normal(0, 1.5, n_lakes),
        "surround_slope_deg": np.clip(rng.normal(27, 7, n_lakes), 5, 55),
        "dam_face_slope_deg": np.clip(rng.normal(19, 6, n_lakes), 3, 40),
        "dam_type": rng.choice(["moraine", "ice", "bedrock"], n_lakes, p=[0.70, 0.08, 0.22]),
    })
    spatial = 0.02 * smooth_field(rng, lat, lon, length_deg=0.8)
    y, drained = lake_response(df, rng, spatial)
    df["log_area_change"] = y
    df["area_2020_km2"] = df["area_2000_km2"] * np.exp(y)
    df["drainage_event"] = drained.astype(int)
    return df


# ---------------------------------------------------------------------------
# 4. Generator B - independent analysis range (different everything)
# ---------------------------------------------------------------------------
def generate_analysis_range(seed: int, n_lakes: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    lat, lon, valley = place_lakes(rng, n_lakes, 45, (38.0, 41.5), (70.0, 79.5), 0.15)
    nv = valley.max() + 1
    v_elev = rng.normal(4680, 280, nv)
    v_glac = _lognormal(rng, 6.0, 0.8, nv)
    f_temp = smooth_field(rng, lat, lon, length_deg=2.0)
    f_mb = smooth_field(rng, lat, lon, length_deg=2.2)
    f_prec = smooth_field(rng, lat, lon, length_deg=2.5)

    df = pd.DataFrame({
        "lake_id": [f"B{i:05d}" for i in range(n_lakes)],
        "lat": lat, "lon": lon, "valley": [f"B-V{v:02d}" for v in valley],
        "elevation_m": v_elev[valley] + rng.normal(0, 160, n_lakes),
        "area_2000_km2": np.clip(_lognormal(rng, 0.06, 0.95, n_lakes), 0.004, 6.0),
        "glacier_area_km2": np.clip(v_glac[valley] * _lognormal(rng, 1.0, 0.8, n_lakes), 0.1, 120),
        "dist_to_glacier_km": np.clip(rng.exponential(0.95, n_lakes), 0.0, 8.0),
        "glacier_slope_deg": np.clip(rng.gamma(4.5, 1.6, n_lakes), 0.5, 30),
        "debris_cover_frac": rng.beta(1.6, 3.4, n_lakes),
        "mass_balance_mwe": -0.36 + 0.13 * f_mb + rng.normal(0, 0.09, n_lakes),
        "temp_trend_c_dec": 0.35 + 0.07 * f_temp + rng.normal(0, 0.035, n_lakes),
        "precip_trend_pct_dec": 0.8 + 3.3 * f_prec + rng.normal(0, 1.8, n_lakes),
        "surround_slope_deg": np.clip(rng.normal(30, 8, n_lakes), 5, 55),
        "dam_face_slope_deg": np.clip(rng.normal(21, 7, n_lakes), 3, 40),
        "dam_type": rng.choice(["moraine", "ice", "bedrock"], n_lakes, p=[0.58, 0.17, 0.25]),
    })
    spatial = 0.02 * smooth_field(rng, lat, lon, length_deg=0.8)
    y, drained = lake_response(df, rng, spatial)
    df["log_area_change"] = y
    df["area_2020_km2"] = df["area_2000_km2"] * np.exp(y)
    df["drainage_event"] = drained.astype(int)
    return df


# ---------------------------------------------------------------------------
# 5. Overview figure
# ---------------------------------------------------------------------------
def overview_figure(train: pd.DataFrame, ana: pd.DataFrame, path: str):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    A, B = "#2a78d6", "#eb6834"
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9,
                         "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
                         "xtick.color": "#898781", "ytick.color": "#898781",
                         "axes.spines.top": False, "axes.spines.right": False})
    fig = plt.figure(figsize=(13, 8), facecolor="#fcfcfb")
    gs = fig.add_gridspec(3, 4, height_ratios=[1.3, 1, 1], hspace=0.55, wspace=0.35)

    ax = fig.add_subplot(gs[0, :])
    ax.scatter(train.lon, train.lat, s=6, c=A, alpha=0.6, lw=0, label=f"A  training range  (n={len(train)})")
    ax.scatter(ana.lon, ana.lat, s=6, c=B, alpha=0.6, lw=0, label=f"B  analysis range  (n={len(ana)})")
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    ax.set_title("Two separate synthetic mountain ranges: no shared lakes, valleys or coordinates",
                 loc="left", fontsize=11, color="#0b0b0b")
    ax.legend(frameon=False, loc="upper right", markerscale=3)
    ax.grid(color="#e1e0d9", lw=0.5)

    cols = ["elevation_m", "area_2000_km2", "dist_to_glacier_km", "debris_cover_frac",
            "mass_balance_mwe", "temp_trend_c_dec", "precip_trend_pct_dec", "glacier_slope_deg"]
    for k, c in enumerate(cols):
        ax = fig.add_subplot(gs[1 + k // 4, k % 4])
        logx = c == "area_2000_km2"
        a, b = train[c], ana[c]
        if logx:
            a, b = np.log10(a), np.log10(b)
        bins = np.linspace(min(a.min(), b.min()), max(a.max(), b.max()), 40)
        ax.hist(a, bins, density=True, color=A, alpha=0.55, label="A")
        ax.hist(b, bins, density=True, color=B, alpha=0.55, label="B")
        ax.set_title(("log10 " if logx else "") + c, fontsize=9, loc="left", color="#0b0b0b")
        ax.set_yticks([])
        if k == 0:
            ax.legend(frameon=False, fontsize=8)
    dt = pd.DataFrame({"A": train.dam_type.value_counts(normalize=True),
                       "B": ana.dam_type.value_counts(normalize=True)}).round(2)
    fig.text(0.01, 0.005, "Dam-type shares  " + "   ".join(
        f"{t}: A {dt.loc[t,'A']:.0%} / B {dt.loc[t,'B']:.0%}" for t in dt.index),
        fontsize=9, color="#52514e")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
def main(argv=None):
    """Run Step 1.

    argv : list of str, optional
        Command-line style options, e.g. ["--analysis-seed", "random"].
        None (the default) reads them from the command line.
    """
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--train-seed", default="2024")
    ap.add_argument("--analysis-seed", default="770077",
                    help="integer, or 'random' for a fresh unseen dataset every run")
    ap.add_argument("--n-train", type=int, default=5000)
    ap.add_argument("--n-analysis", type=int, default=1500)
    # parse_known_args ignores the extra "-f <kernel>.json" argument that
    # Jupyter / ArcGIS Pro notebooks pass, so the script runs in a notebook too.
    args = ap.parse_known_args(argv)[0]

    os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
    tseed = int(args.train_seed)
    aseed = (int(np.random.SeedSequence().entropy % 2**31)
             if args.analysis_seed == "random" else int(args.analysis_seed))
    if aseed == tseed:
        raise SystemExit("Analysis seed must differ from training seed.")

    train = generate_training_range(tseed, args.n_train)
    ana = generate_analysis_range(aseed, args.n_analysis)

    train.to_csv(f"{DATA_DIR}/train_lakes.csv", index=False)
    feat_cols = ["lake_id", "lat", "lon", "valley"] + FEATURES
    ana[feat_cols].to_csv(f"{DATA_DIR}/analysis_features.csv", index=False)
    ana[["lake_id", "log_area_change", "area_2020_km2", "drainage_event"]].to_csv(
        f"{DATA_DIR}/analysis_answer_key.csv", index=False)

    meta = {"train_seed": tseed, "analysis_seed": aseed,
            "n_train": len(train), "n_analysis": len(ana),
            "train_bbox": [27.5, 31.0, 80.5, 90.0], "analysis_bbox": [38.0, 41.5, 70.0, 79.5],
            "features": FEATURES}
    with open(f"{DATA_DIR}/generation_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    overview_figure(train, ana, f"{FIG_DIR}/step1_datasets_overview.png")

    print("STEP 1 complete")
    print(f"  Training range A : {len(train)} lakes, seed {tseed}, "
          f"median change {np.median(train.log_area_change):+.3f}, drainage events {train.drainage_event.sum()}")
    print(f"  Analysis range B : {len(ana)} lakes, seed {aseed}, "
          f"(truth stored separately in analysis_answer_key.csv)")
    print("  Feature means (A vs B):")
    num = [c for c in FEATURES if c != "dam_type"]
    print(pd.DataFrame({"A": train[num].mean(), "B": ana[num].mean()}).round(3).to_string())


if __name__ == "__main__":
    main()

STEP 1 complete
  Training range A : 5000 lakes, seed 2024, median change +0.267, drainage events 32
  Analysis range B : 1500 lakes, seed 770077, (truth stored separately in analysis_answer_key.csv)
  Feature means (A vs B):
                             A         B
elevation_m           4913.917  4682.398
area_2000_km2            0.130     0.092
glacier_area_km2        11.639     9.305
dist_to_glacier_km       1.190     0.929
glacier_slope_deg        5.981     7.273
debris_cover_frac        0.426     0.322
mass_balance_mwe        -0.449    -0.360
temp_trend_c_dec         0.301     0.349
precip_trend_pct_dec    -1.994     0.728
surround_slope_deg      27.031    29.976
dam_face_slope_deg      18.992    20.910


In [7]:
"""
STEP 2 - Train the lake-change model and calibrate split conformal intervals
============================================================================

Input : data/train_lakes.csv   (training range A only - never touches range B)
Output: models/glof_cp_bundle.joblib, outputs/step2_metrics.json,
        figures/step2_training_diagnostics.png

Pipeline
--------
1. Spatial blocking. Lakes are grouped into 0.5° x 0.5° grid cells. Whole cells
   go to one of three sets so neighbouring lakes never sit on both sides of a
   split:  proper-training 60 %  |  calibration 20 %  |  test 20 %.
2. Point model f(x): histogram gradient boosting on PHYSICAL features only.
   Latitude/longitude are deliberately NOT features: the model must learn
   processes that transfer to a different mountain range, not memorise places.
3. Difficulty model sigma(x): a second booster trained on spatially
   out-of-fold absolute residuals. It tells the conformal step which lakes are
   inherently harder (glacier-contact, ice-dammed, tiny lakes).
4. Split conformal prediction (normalized score):
        s_i = |y_i - f(x_i)| / sigma(x_i)        on the calibration set
        q   = the k-th smallest s_i
        interval(x) = f(x) +/- q * sigma(x)
   Standard rank k = ceil((n+1)(1-alpha)) gives P(y in interval) >= 1-alpha
   on average over calibration draws. Because the goal here is MORE than
   90 % coverage, the default is the training-conditional rank (--delta 0.05,
   Vovk 2012): coverage >= 90 % with 95 % confidence for this calibration
   set. Use --delta none for the textbook version.
   A plain (un-normalized) split-CP interval is computed too, as a baseline.
5. Checks on the untouched test blocks: R², MAE, coverage overall and per
   group, plus 300 random re-draws of the calibration/test block partition to
   show the coverage distribution, not just one lucky number.

Run:  python step2_train_calibrate.py [--alpha 0.1] [--block-deg 0.5]
In a Jupyter / ArcGIS Pro notebook:
      main()                                   (defaults)
      main(["--alpha", "0.05", "--delta", "none"])
"""
from __future__ import annotations

import argparse
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

DAM_TYPES = ["moraine", "ice", "bedrock"]
NUMERIC = ["elevation_m", "area_2000_km2", "glacier_area_km2", "dist_to_glacier_km",
           "glacier_slope_deg", "debris_cover_frac", "mass_balance_mwe",
           "temp_trend_c_dec", "precip_trend_pct_dec", "surround_slope_deg",
           "dam_face_slope_deg"]
TARGET = "log_area_change"


# ---------------------------------------------------------------------------
# Shared helpers (Step 3 imports these)
# ---------------------------------------------------------------------------
def make_X(df: pd.DataFrame) -> pd.DataFrame:
    """Model matrix: physical features + log area + one-hot dam type."""
    X = df[NUMERIC].copy()
    X["log_area_2000"] = np.log(df["area_2000_km2"])
    X["glacier_contact"] = np.exp(-df["dist_to_glacier_km"] / 0.8)
    for t in DAM_TYPES:
        X[f"dam_{t}"] = (df["dam_type"] == t).astype(float)
    return X


def spatial_blocks(lat, lon, deg):
    return (np.floor(np.asarray(lat) / deg).astype(int).astype(str) + "_"
            + np.floor(np.asarray(lon) / deg).astype(int).astype(str))


def conformal_rank(n: int, alpha: float, delta: float | None = None) -> int:
    """Rank k of the calibration score used as threshold.

    delta=None : standard split CP, k = ceil((n+1)(1-alpha)).
                 Coverage >= 1-alpha *on average* over calibration sets.
    delta=0.05 : training-conditional ("PAC") split CP (Vovk 2012).
                 Coverage of an exchangeable new lake given THIS calibration
                 set is Beta(k, n+1-k); pick the smallest k with
                 P(coverage >= 1-alpha) >= 1-delta. Slightly wider intervals,
                 but >= 90 % coverage holds with 95 % confidence instead of
                 only on average.
    """
    if delta is None:
        return int(np.ceil((n + 1) * (1 - alpha)))
    from scipy.stats import beta
    ks = np.arange(1, n + 1)
    ok = beta.sf(1 - alpha, ks, n + 1 - ks) >= 1 - delta
    return int(ks[ok][0]) if ok.any() else n + 1


def conformal_quantile(scores: np.ndarray, alpha: float, delta: float | None = None) -> float:
    """Finite-sample-corrected conformal threshold of calibration scores."""
    n = len(scores)
    k = conformal_rank(n, alpha, delta)
    if k > n:
        return np.inf
    return float(np.sort(scores)[k - 1])


def predict_mean_sigma(bundle: dict, X: pd.DataFrame):
    mu = bundle["mean_model"].predict(X)
    sigma = np.exp(bundle["sigma_model"].predict(X))
    return mu, np.maximum(sigma, bundle["sigma_floor"])


def base_model():
    return HistGradientBoostingRegressor(
        max_iter=900, learning_rate=0.04, max_leaf_nodes=24, min_samples_leaf=20,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=40, random_state=0)


def sigma_model():
    return HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.04, max_leaf_nodes=12, min_samples_leaf=40,
        l2_regularization=2.0, random_state=1)


# ---------------------------------------------------------------------------
def block_split(blocks, rng, fracs=(0.6, 0.2, 0.2)):
    ub = rng.permutation(np.unique(blocks))
    n1 = int(round(fracs[0] * len(ub))); n2 = n1 + int(round(fracs[1] * len(ub)))
    return (np.isin(blocks, ub[:n1]), np.isin(blocks, ub[n1:n2]), np.isin(blocks, ub[n2:]))


def oof_residuals(X, y, groups, n_splits=5):
    oof = np.zeros(len(y))
    for tr, va in GroupKFold(n_splits=n_splits).split(X, y, groups):
        m = clone(base_model()).fit(X.iloc[tr], y[tr])
        oof[va] = y[va] - m.predict(X.iloc[va])
    return oof


def diagnostics_figure(test_df, lo, hi, mu, rep_cov, group_cov, alpha, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9,
                         "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
                         "xtick.color": "#898781", "ytick.color": "#898781",
                         "axes.spines.top": False, "axes.spines.right": False})
    BLUE, RED, INK = "#2a78d6", "#d03b3b", "#0b0b0b"
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6), facecolor="#fcfcfb",
                            gridspec_kw={"width_ratios": [1.5, 1, 1]})

    y = test_df[TARGET].to_numpy()
    order = np.argsort(mu)
    sub = order[np.linspace(0, len(order) - 1, min(160, len(order))).astype(int)]
    cov = (y >= lo) & (y <= hi)
    ax = axs[0]
    xs = np.arange(len(sub))
    ax.fill_between(xs, lo[sub], hi[sub], color=BLUE, alpha=0.18, lw=0, label=f"{1-alpha:.0%} conformal interval")
    ax.plot(xs, mu[sub], color=BLUE, lw=2, label="prediction")
    ax.scatter(xs[cov[sub]], y[sub][cov[sub]], s=10, color=INK, lw=0, label="observed, inside")
    ax.scatter(xs[~cov[sub]], y[sub][~cov[sub]], s=24, color=RED, marker="x", label="observed, outside")
    ax.set_xlabel("Test lakes (sorted by prediction)"); ax.set_ylabel("log(A2020 / A2000)")
    ax.set_title("Held-out test blocks: prediction intervals", loc="left", color=INK)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax.grid(axis="y", color="#e1e0d9", lw=0.5)

    ax = axs[1]
    names, vals, ns = zip(*group_cov)
    ypos = np.arange(len(names))
    ax.barh(ypos, vals, color=BLUE, height=0.6)
    ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1)
    for i, (v, n) in enumerate(zip(vals, ns)):
        ax.text(v + 0.005, i, f"{v:.1%}  (n={n})", va="center", fontsize=8, color="#52514e")
    ax.set_yticks(ypos, names); ax.set_xlim(0.6, 1.08); ax.invert_yaxis()
    ax.set_xlabel("Empirical coverage"); ax.set_title("Coverage by group (dashed = target)", loc="left", color=INK)

    ax = axs[2]
    ax.hist(rep_cov, bins=30, color=BLUE, alpha=0.85)
    ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1)
    ax.axvline(np.mean(rep_cov), color=INK, lw=1.5)
    ax.set_xlabel("Test coverage"); ax.set_yticks([])
    ax.set_title(f"300 re-drawn calib/test block splits\nmean {np.mean(rep_cov):.3f}", loc="left", color=INK)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def main(argv=None):
    """Run Step 2.

    argv : list of str, optional
        Command-line style options, e.g. ["--alpha", "0.05"].
        None (the default) reads them from the command line.
    """
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--alpha", type=float, default=0.10)
    ap.add_argument("--block-deg", type=float, default=0.5)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--delta", default="0.05",
                    help="confidence slack for training-conditional CP, or 'none' for standard split CP")
    # parse_known_args ignores the extra "-f <kernel>.json" argument that
    # Jupyter / ArcGIS Pro notebooks pass, so the script runs in a notebook too.
    args = ap.parse_known_args(argv)[0]
    delta = None if str(args.delta).lower() == "none" else float(args.delta)
    os.makedirs("models", exist_ok=True); os.makedirs("outputs", exist_ok=True)
    os.makedirs("figures", exist_ok=True)

    df = pd.read_csv("data/train_lakes.csv")
    X_all, y_all = make_X(df), df[TARGET].to_numpy()
    blocks = spatial_blocks(df.lat, df.lon, args.block_deg)
    rng = np.random.default_rng(args.seed)
    tr, ca, te = block_split(blocks, rng)
    print(f"Spatial blocks: {len(np.unique(blocks))} cells of {args.block_deg}° | "
          f"lakes train/calib/test = {tr.sum()}/{ca.sum()}/{te.sum()}")

    # --- 1. point model -----------------------------------------------------
    Xtr, ytr = X_all[tr], y_all[tr]
    mean_model = clone(base_model()).fit(Xtr, ytr)

    # --- 2. difficulty model on spatially out-of-fold residuals -------------
    r_oof = oof_residuals(Xtr.reset_index(drop=True), ytr, blocks[tr])
    sig_model = clone(sigma_model()).fit(Xtr, np.log(np.abs(r_oof) + 1e-3))
    sigma_floor = float(np.quantile(np.abs(r_oof), 0.05) + 1e-3)
    bundle = {"mean_model": mean_model, "sigma_model": sig_model, "sigma_floor": sigma_floor}

    # --- 3. split conformal calibration ------------------------------------
    mu_c, sg_c = predict_mean_sigma(bundle, X_all[ca])
    res_c = np.abs(y_all[ca] - mu_c)
    s_norm, s_abs = res_c / sg_c, res_c
    q_norm, q_abs = conformal_quantile(s_norm, args.alpha, delta), conformal_quantile(s_abs, args.alpha, delta)
    q_norm_marg = conformal_quantile(s_norm, args.alpha)
    k = conformal_rank(len(s_norm), args.alpha, delta)
    print(f"Calibration: n={len(s_norm)}, threshold rank k={k} "
          f"(quantile level {k/len(s_norm):.4f}; standard split CP would use "
          f"{conformal_rank(len(s_norm), args.alpha)})")

    # --- 4. evaluate on untouched test blocks ------------------------------
    mu_t, sg_t = predict_mean_sigma(bundle, X_all[te])
    y_t = y_all[te]
    lo, hi = mu_t - q_norm * sg_t, mu_t + q_norm * sg_t
    cov_norm = float(np.mean((y_t >= lo) & (y_t <= hi)))
    cov_abs = float(np.mean(np.abs(y_t - mu_t) <= q_abs))
    r2, mae = r2_score(y_t, mu_t), mean_absolute_error(y_t, mu_t)

    test_df = df[te].reset_index(drop=True)
    covered = (y_t >= lo) & (y_t <= hi)
    contact = np.exp(-test_df.dist_to_glacier_km / 0.8)
    groups = [(f"dam: {t}", (test_df.dam_type == t).to_numpy()) for t in DAM_TYPES]
    groups += [("glacier contact", (contact > 0.5).to_numpy()), ("detached", (contact <= 0.5).to_numpy()),
               ("small lake < 0.05 km²", (test_df.area_2000_km2 < 0.05).to_numpy()),
               ("large lake ≥ 0.05 km²", (test_df.area_2000_km2 >= 0.05).to_numpy())]
    group_cov = [(n, float(covered[m].mean()), int(m.sum())) for n, m in groups if m.sum() > 0]

    # --- 5. robustness: re-draw calibration / test partition many times -----
    pool = ca | te
    mu_p, sg_p = predict_mean_sigma(bundle, X_all[pool])
    s_p, y_p, b_p = np.abs(y_all[pool] - mu_p) / sg_p, y_all[pool], blocks[pool]
    ub = np.unique(b_p); rep_cov = []
    for _ in range(300):
        cb = rng.permutation(ub)[: len(ub) // 2]
        mc = np.isin(b_p, cb)
        q = conformal_quantile(s_p[mc], args.alpha, delta)
        rep_cov.append(float(np.mean(s_p[~mc] <= q)))
    rep_cov = np.array(rep_cov)

    # --- 6. save everything Step 3 needs (no training data rows besides the
    #        calibration features/scores, which conformal prediction requires)
    X_cal = X_all[ca].reset_index(drop=True)
    bundle.update({
        "alpha": args.alpha, "delta": delta, "q_norm": q_norm, "q_abs": q_abs,
        "q_norm_marginal": q_norm_marg,
        "cal_scores_norm": s_norm, "cal_scores_abs": s_abs, "X_cal": X_cal,
        "feature_columns": list(X_all.columns),
        "train_feature_range": {c: (float(X_all[c].quantile(0.005)), float(X_all[c].quantile(0.995)))
                                for c in NUMERIC},
        "block_deg": args.block_deg,
    })
    joblib.dump(bundle, "models/glof_cp_bundle.joblib")

    metrics = {
        "alpha": args.alpha, "delta": delta, "target_coverage": 1 - args.alpha,
        "n_train": int(tr.sum()), "n_calibration": int(ca.sum()), "n_test": int(te.sum()),
        "test_R2": round(r2, 4), "test_MAE_log": round(mae, 4),
        "coverage_normalized_cp": round(cov_norm, 4),
        "coverage_plain_split_cp": round(cov_abs, 4),
        "mean_width_normalized_cp": round(float(np.mean(hi - lo)), 4),
        "width_plain_split_cp": round(2 * q_abs, 4),
        "repeated_split_coverage_mean": round(float(rep_cov.mean()), 4),
        "repeated_split_coverage_p05": round(float(np.quantile(rep_cov, 0.05)), 4),
        "group_coverage": {n: round(v, 4) for n, v, _ in group_cov},
    }
    with open("outputs/step2_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    diagnostics_figure(test_df, lo, hi, mu_t, rep_cov, group_cov, args.alpha,
                       "figures/step2_training_diagnostics.png")

    print("\nSTEP 2 complete - model trained on range A only")
    print(f"  Test R²                         : {r2:.3f}")
    print(f"  Test MAE (log ratio)            : {mae:.3f}")
    print(f"  Coverage, normalized split CP   : {cov_norm:.3f}  (target ≥ {1-args.alpha:.2f}), "
          f"mean width {np.mean(hi-lo):.3f}")
    print(f"  Coverage, plain split CP        : {cov_abs:.3f}, width {2*q_abs:.3f}")
    print(f"  300 re-drawn splits             : mean {rep_cov.mean():.3f}, 5th pct {np.quantile(rep_cov, 0.05):.3f}")
    for n, v, k in group_cov:
        print(f"    {n:<24s} {v:.3f}  (n={k})")


if __name__ == "__main__":
    main()

Spatial blocks: 112 cells of 0.5° | lakes train/calib/test = 2798/916/1286
Calibration: n=916, threshold rank k=840 (quantile level 0.9170; standard split CP would use 826)

STEP 2 complete - model trained on range A only
  Test R²                         : 0.911
  Test MAE (log ratio)            : 0.068
  Coverage, normalized split CP   : 0.926  (target ≥ 0.90), mean width 0.335
  Coverage, plain split CP        : 0.925, width 0.309
  300 re-drawn splits             : mean 0.915, 5th pct 0.888
    dam: moraine             0.928  (n=891)
    dam: ice                 0.873  (n=110)
    dam: bedrock             0.940  (n=285)
    glacier contact          0.935  (n=465)
    detached                 0.921  (n=821)
    small lake < 0.05 km²    0.918  (n=403)
    large lake ≥ 0.05 km²    0.930  (n=883)


In [9]:
"""
STEP 3 - Analyse the independent synthetic range B with the frozen model
========================================================================

Input : models/glof_cp_bundle.joblib       (from Step 2, never retrained here)
        data/analysis_features.csv          (range B - features only)
        data/analysis_answer_key.csv        (read ONLY in the final validation block)
Output: outputs/analysis_predictions.csv    per-lake prediction, interval, risk
        outputs/high_risk_zones.csv         ranked 0.25° zones
        outputs/step3_validation.json
        figures/step3_spatial_uncertainty_maps.png
        figures/step3_validation.png

What happens
------------
1. Predict f(x) and sigma(x) for every range-B lake with the frozen models.
2. Conformal intervals, three versions (choose with --method):
   standard          normalized split CP - the single q from Step 2;
   mondrian          (DEFAULT) a separate q per dam type, so ice-dammed lakes -
                     the most GLOF-prone, and three times more common in
                     range B than in training - get their own 90 % guarantee
                     instead of borrowing it from moraine lakes;
   mondrian-weighted mondrian + covariate-shift weights (Tibshirani et al.
                     2019): a classifier separating calibration lakes from
                     range-B lakes (features only, no labels) estimates the
                     likelihood ratio w(x) and calibration scores are
                     re-weighted per lake. Theoretically the right fix for
                     distribution shift, but with strong shift the effective
                     calibration size collapses and intervals blow up, so
                     here it is reported as an option and the weights are
                     kept as a per-lake "how unfamiliar is this lake" flag.
   The prediction model itself is never touched.
3. Applicability check: how many features of each lake fall outside the
   range seen in training (extrapolation flag).
4. GLOF susceptibility index (0-1) from future lake size, growth, glacier
   contact, surrounding slope (mass-movement trigger), dam-face steepness and
   dam type. The index rises monotonically with lake change y, so evaluating
   it at the interval ends gives a risk interval [risk_lo, risk_hi] that keeps
   the 90 % conformal coverage.
5. Risk classes:  High-confident (risk_lo >= T), High-possible
   (risk_hi >= T > risk_lo), Moderate, Low. "Possible" lakes are exactly the
   ones where uncertainty decides the call - the ones to look at next.
6. Spatial maps: kernel-smoothed predicted change, interval width
   (uncertainty), upper-bound risk, and ranked high-risk zones.
7. Validation against the hidden answer key.

Run:  python step3_analyse_new_region.py [--risk-threshold 0.5]
In a Jupyter / ArcGIS Pro notebook:
      main()                                   (defaults)
      main(["--method", "standard", "--replicates", "0"])
Steps 1 and 2 must either be saved as .py files in the same folder or have
been run in earlier cells of the same notebook session.
"""
from __future__ import annotations

import argparse
import importlib
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def _from_earlier_step(module: str, names: list[str]) -> list:
    """Get functions/constants from an earlier pipeline step.

    Works when that step is saved as <module>.py in the working folder, and
    also in a Jupyter / ArcGIS Pro notebook where the step was pasted into an
    earlier cell and run (its functions then live in the notebook session).
    """
    try:
        mod = importlib.import_module(module)
        return [getattr(mod, n) for n in names]
    except ModuleNotFoundError as err:
        if err.name != module:      # the file exists but one of ITS imports failed
            raise
    import __main__
    session = vars(__main__)
    if any(n not in session for n in names):
        raise ModuleNotFoundError(
            f"Can't find {module}. Either save it as {module}.py in the working "
            f"folder ({os.getcwd()}), or run that step's cell earlier in this "
            f"notebook session.", name=module)
    return [session[n] for n in names]


NUMERIC, conformal_quantile, make_X, predict_mean_sigma = _from_earlier_step(
    "step2_train_calibrate", ["NUMERIC", "conformal_quantile", "make_X", "predict_mean_sigma"])

DAM_WEIGHT = {"moraine": 1.0, "ice": 0.9, "bedrock": 0.25}


# ---------------------------------------------------------------------------
# Covariate-shift weighted conformal prediction
# ---------------------------------------------------------------------------
def shift_weights(X_cal: pd.DataFrame, X_new: pd.DataFrame, clip=(0.1, 10.0)):
    """Likelihood ratio w(x) = p_new(x)/p_cal(x) via a probabilistic classifier."""
    Z = pd.concat([X_cal, X_new], ignore_index=True)
    lab = np.r_[np.zeros(len(X_cal)), np.ones(len(X_new))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000))
    clf.fit(Z, lab)
    p_cal = np.clip(clf.predict_proba(X_cal)[:, 1], 1e-4, 1 - 1e-4)
    p_new = np.clip(clf.predict_proba(X_new)[:, 1], 1e-4, 1 - 1e-4)
    ratio = len(X_cal) / len(X_new)
    w_cal = np.clip(p_cal / (1 - p_cal) * ratio, *clip)
    w_new = np.clip(p_new / (1 - p_new) * ratio, *clip)
    return w_cal, w_new


def mondrian_q(scores, g_cal, g_new, alpha, delta=None, min_group=30):
    """Split CP threshold computed separately for each group (dam type)."""
    q = np.full(len(g_new), conformal_quantile(scores, alpha, delta))
    for g in np.unique(g_new):
        mc = g_cal == g
        if mc.sum() >= min_group:
            q[g_new == g] = conformal_quantile(scores[mc], alpha, delta)
    return q


def mondrian_weighted_q(scores, g_cal, g_new, w_cal, w_new, alpha, min_group=30):
    """Weighted CP calibrated within each group (dam type); groups with too
    few calibration lakes fall back to the pooled calibration set."""
    q = weighted_conformal_q(scores, w_cal, w_new, alpha)
    for g in np.unique(g_new):
        mc, mn = g_cal == g, g_new == g
        if mc.sum() >= min_group:
            q[mn] = weighted_conformal_q(scores[mc], w_cal[mc], w_new[mn], alpha)
    return q


def weighted_conformal_q(scores, w_cal, w_new, alpha):
    """Per-test-point quantile of the weighted score distribution
    sum_i w_i δ(s_i) + w_new δ(+inf), normalised."""
    order = np.argsort(scores)
    s, cw = scores[order], np.cumsum(w_cal[order])
    W = cw[-1]
    idx = np.searchsorted(cw, (1 - alpha) * (W + w_new), side="left")
    q = np.full(len(w_new), np.inf)
    ok = idx < len(s)
    q[ok] = s[idx[ok]]
    return q


# ---------------------------------------------------------------------------
# GLOF susceptibility index
# ---------------------------------------------------------------------------
def glof_risk(df: pd.DataFrame, y: np.ndarray) -> np.ndarray:
    a2 = df["area_2000_km2"].to_numpy() * np.exp(y)
    s_size = np.clip(np.log10(a2 / 0.02) / np.log10(50), 0, 1)       # 0.02 km² -> 0, 1 km² -> 1
    s_growth = np.clip(y / 0.8, 0, 1)                                  # +120 % in 20 yr -> 1
    contact = np.exp(-df["dist_to_glacier_km"].to_numpy() / 0.8)
    s_trigger = 0.6 * contact + 0.4 * np.clip((df["surround_slope_deg"].to_numpy() - 20) / 25, 0, 1)
    s_dam = np.clip((df["dam_face_slope_deg"].to_numpy() - 10) / 20, 0, 1)
    dam_w = df["dam_type"].map(DAM_WEIGHT).to_numpy()
    return dam_w * (0.35 * s_size + 0.25 * s_growth + 0.25 * s_trigger + 0.15 * s_dam)


def risk_class(r_lo, r_hi, t_high, t_mod):
    return np.select([r_lo >= t_high, r_hi >= t_high, r_hi >= t_mod],
                     ["High (confident)", "High (possible)", "Moderate"], "Low")


# ---------------------------------------------------------------------------
# Spatial smoothing onto a grid
# ---------------------------------------------------------------------------
def kernel_grid(lat, lon, values, bbox, res=0.04, bw=0.12, min_density=0.6):
    la = np.arange(bbox[0], bbox[1] + res, res)
    lo = np.arange(bbox[2], bbox[3] + res, res)
    G_lo, G_la = np.meshgrid(lo, la)
    coslat = np.cos(np.deg2rad(np.mean(bbox[:2])))
    d2 = ((G_la.ravel()[:, None] - lat[None, :]) ** 2
          + ((G_lo.ravel()[:, None] - lon[None, :]) * coslat) ** 2)
    K = np.exp(-0.5 * d2 / bw ** 2)
    dens = K.sum(1)
    Z = (K @ values) / np.maximum(dens, 1e-12)
    Z[dens < min_density] = np.nan
    return G_lo, G_la, Z.reshape(G_la.shape), dens.reshape(G_la.shape)


def _cmap(name, colors):
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list(name, colors)


def _style():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9,
                         "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
                         "xtick.color": "#898781", "ytick.color": "#898781",
                         "axes.spines.top": False, "axes.spines.right": False})
    return plt


CLASS_STYLE = {  # status palette + marker shape, so class never relies on colour alone
    "High (confident)": ("#d03b3b", "^", 34),
    "High (possible)": ("#ec835a", "D", 20),
    "Moderate": ("#fab219", "o", 8),
    "Low": ("#c3c2b7", "o", 4),
}


def map_figure(df, bbox, zones, path):
    plt = _style()
    INK = "#0b0b0b"
    seq_blue = _cmap("b", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
    seq_orange = _cmap("o", ["#fde3d6", "#f5ab88", "#eb6834", "#b8461c", "#6e2a10"])
    div = _cmap("d", ["#104281", "#3987e5", "#f0efec", "#e66767", "#9e2222"])

    lat, lon = df.lat.to_numpy(), df.lon.to_numpy()
    fig, axs = plt.subplots(2, 2, figsize=(15, 8.8), facecolor="#fcfcfb")
    panels = [
        ("pct_change_pred", "(a) Predicted lake-area change 2000→2020 (%)", div, None),
        ("interval_width_log", "(b) Spatial uncertainty: 90 % interval width (log ratio)", seq_blue, None),
        ("risk_hi", "(c) GLOF susceptibility, upper bound of 90 % risk interval", seq_orange, (0, 0.8)),
    ]
    for ax, (col, title, cmap, lim) in zip(axs.ravel()[:3], panels):
        v = df[col].to_numpy()
        if col == "pct_change_pred":
            v = np.clip(v, -100, 300)
        Gx, Gy, Z, _ = kernel_grid(lat, lon, v, bbox)
        if col == "pct_change_pred":
            m = np.nanmax(np.abs(np.nanpercentile(Z, [2, 98])))
            vmin, vmax = -m, m
        elif lim:
            vmin, vmax = lim
        else:
            vmin, vmax = np.nanpercentile(Z, [2, 98])
        pc = ax.pcolormesh(Gx, Gy, Z, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        ax.scatter(lon, lat, s=1.5, c="#52514e", alpha=0.35, lw=0)
        fig.colorbar(pc, ax=ax, fraction=0.03, pad=0.01)
        ax.set_title(title, loc="left", color=INK, fontsize=10)

    ax = axs[1, 1]
    hi_mask = df.risk_class.str.startswith("High").to_numpy().astype(float)
    Gx, Gy, Z, dens = kernel_grid(lat, lon, hi_mask * df.risk_hi.to_numpy(), bbox, bw=0.1)
    Zc = Z * np.minimum(dens / 3.0, 1.0)
    Zc = np.nan_to_num(Zc)
    lv = np.nanpercentile(Zc[Zc > 0], [80, 93]) if (Zc > 0).any() else [1, 2]
    ax.contourf(Gx, Gy, Zc, levels=[lv[0], lv[1], np.inf], colors=["#fbe0d4", "#f5b9a3"], alpha=0.9)
    for cls in ["Low", "Moderate", "High (possible)", "High (confident)"]:
        m = (df.risk_class == cls).to_numpy()
        c, mk, s = CLASS_STYLE[cls]
        ax.scatter(lon[m], lat[m], s=s, c=c, marker=mk, lw=0.4 if cls.startswith("High") else 0,
                   edgecolors="#fcfcfb", label=f"{cls}  (n={m.sum()})", zorder=3)
    for _, z in zones.head(5).iterrows():
        ax.annotate(f"Z{int(z['rank'])}", (z.lon_center, z.lat_center), xytext=(6, 6),
                    textcoords="offset points", fontsize=9, color=INK, weight="bold", zorder=4)
    ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=4, markerscale=1.4)
    ax.set_title("(d) High-risk zones (shaded) and lake risk class; Z1 = highest-ranked zone",
                 loc="left", color=INK, fontsize=10)
    for ax in axs.ravel():
        ax.set_xlim(bbox[2], bbox[3]); ax.set_ylim(bbox[0], bbox[1])
        ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(bbox[:2]))))
        ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    fig.suptitle("Synthetic Range B - lakes never seen in training, frozen model + split conformal prediction",
                 x=0.01, ha="left", fontsize=12, color=INK)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def validation_figure(res, df, alpha, t_high, path):
    plt = _style()
    INK, BLUE, GRAY = "#0b0b0b", "#2a78d6", "#898781"
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6), facecolor="#fcfcfb",
                            gridspec_kw={"width_ratios": [1, 1, 1.4]})
    ax = axs[0]
    names = METHODS
    vals = [res["coverage"][k] for k in names]
    cols = [BLUE if k == res["method_used"] else GRAY for k in names]
    ax.barh(range(len(names)), vals, color=cols, height=0.55)
    rep = res.get("replicates")
    for i, (k, v) in enumerate(zip(names, vals)):
        lab = f"{v:.1%}  width {res['mean_width'][k]:.2f}"
        if rep:
            r = rep[k]
            ax.plot([r["coverage_min"], r["coverage_max"]], [i, i], color=INK, lw=1.2)
            ax.plot([r["coverage_mean"]], [i], "|", color=INK, ms=10, mew=2)
            lab += f"\n{rep['n']} replicates: mean {r['coverage_mean']:.1%}"
        ax.text(0.955, i, lab, va="center", fontsize=8, color="#52514e")
    ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1)
    ax.set_yticks(range(len(names)), names); ax.set_xlim(0.80, 1.06); ax.invert_yaxis()
    ax.set_title("Coverage on range B (bar) and range over\nreplicate datasets (black line); dashed = 90 % target",
                 loc="left", color=INK)

    ax = axs[1]
    g = res["coverage_by_group"]
    ax.barh(range(len(g)), list(g.values()), color=BLUE, height=0.55)
    for i, (k, v) in enumerate(g.items()):
        ax.text(v + 0.005, i, f"{v:.1%}", va="center", color="#52514e")
    ax.axvline(1 - alpha, color="#52514e", ls="--", lw=1)
    ax.set_yticks(range(len(g)), list(g.keys())); ax.set_xlim(0.6, 1.08); ax.invert_yaxis()
    ax.set_title(f"Coverage by group ({res['method_used']})", loc="left", color=INK)

    ax = axs[2]
    top = df.sort_values("risk_point", ascending=False).head(60).reset_index(drop=True)
    x = np.arange(len(top))
    ax.vlines(x, top.risk_lo, top.risk_hi, color=BLUE, lw=2, alpha=0.55, label="90 % risk interval")
    ax.scatter(x, top.risk_point, s=12, color=BLUE, zorder=3, label="predicted risk")
    inside = (top.risk_true >= top.risk_lo) & (top.risk_true <= top.risk_hi)
    ax.scatter(x[inside], top.risk_true[inside], s=16, color=INK, marker="_", lw=2, zorder=4, label="true risk (answer key)")
    ax.scatter(x[~inside], top.risk_true[~inside], s=26, color="#d03b3b", marker="x", zorder=4, label="true risk outside")
    ax.axhline(t_high, color="#52514e", ls="--", lw=1)
    ax.text(len(top) - 1, t_high + 0.01, "high-risk threshold", ha="right", fontsize=8, color="#52514e")
    ax.set_xlabel("60 highest-ranked lakes"); ax.set_ylabel("GLOF susceptibility index")
    ax.set_title("Risk intervals vs hidden truth", loc="left", color=INK)
    ax.legend(frameon=False, fontsize=8, loc="lower left")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


METHODS = ["standard", "mondrian", "mondrian-weighted"]


def interval_pipeline(bundle, df):
    """Frozen prediction + per-lake conformal quantile q for every method.
    Interval = mu +/- q * sigma."""
    X = make_X(df)[bundle["feature_columns"]]
    mu, sg = predict_mean_sigma(bundle, X)
    sc, a = bundle["cal_scores_norm"], bundle["alpha"]
    Xc = bundle["X_cal"]
    g_cal = np.select([Xc.dam_ice == 1, Xc.dam_bedrock == 1], ["ice", "bedrock"], "moraine")
    g_new = df.dam_type.to_numpy()
    w_cal, w_new = shift_weights(Xc, X)
    q = {"standard": np.full(len(df), bundle["q_norm"]),
         "mondrian": mondrian_q(sc, g_cal, g_new, a, bundle.get("delta")),
         "mondrian-weighted": mondrian_weighted_q(sc, g_cal, g_new, w_cal, w_new, a)}
    cap = 3 * np.max(sc)            # finite cap when a weighted quantile is +inf
    q = {k: np.minimum(v, cap) for k, v in q.items()}
    return mu, sg, q, w_cal, w_new


def replicate_check(bundle, n_rep, base_seed):
    """Evaluation only: generate n_rep MORE independent range-B datasets with
    fresh seeds and check coverage on each. The conformal guarantee is about
    the average over datasets, so one dataset can land a point above or below."""
    (generate_analysis_range,) = _from_earlier_step("step1_generate_data", ["generate_analysis_range"])
    seeds = np.random.SeedSequence(base_seed).generate_state(n_rep)
    rows = []
    for sd in seeds:
        d = generate_analysis_range(int(sd))
        mu, sg, q, _, _ = interval_pipeline(bundle, d)
        s = np.abs(d.log_area_change.to_numpy() - mu) / sg
        row = {"seed": int(sd), "R2": r2_score(d.log_area_change, mu)}
        row.update({f"coverage_{k}": float(np.mean(s <= v)) for k, v in q.items()})
        row.update({f"width_{k}": float(np.mean(2 * v * sg)) for k, v in q.items()})
        rows.append(row)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
def main(argv=None):
    """Run Step 3.

    argv : list of str, optional
        Command-line style options, e.g. ["--risk-threshold", "0.5"].
        None (the default) reads them from the command line.
    """
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--method", choices=METHODS, default="mondrian")
    ap.add_argument("--risk-threshold", type=float, default=0.60, help="high-risk threshold T")
    ap.add_argument("--moderate-threshold", type=float, default=0.40)
    ap.add_argument("--zone-deg", type=float, default=0.25)
    ap.add_argument("--replicates", type=int, default=20,
                    help="extra independent range-B datasets for the coverage robustness check (0 = skip)")
    # parse_known_args ignores the extra "-f <kernel>.json" argument that
    # Jupyter / ArcGIS Pro notebooks pass, so the script runs in a notebook too.
    args = ap.parse_known_args(argv)[0]
    os.makedirs("outputs", exist_ok=True); os.makedirs("figures", exist_ok=True)

    bundle = joblib.load("models/glof_cp_bundle.joblib")
    meta = json.load(open("data/generation_meta.json"))
    df = pd.read_csv("data/analysis_features.csv")
    assert "log_area_change" not in df.columns, "analysis features must not contain the target"
    alpha = bundle["alpha"]

    # 1. frozen predictions -------------------------------------------------
    # 2. standard split CP (a) and Mondrian shift-weighted CP (b) -----------
    mu, sg, q, w_cal, w_new = interval_pipeline(bundle, df)
    lo, hi = mu - q[args.method] * sg, mu + q[args.method] * sg
    ess = w_cal.sum() ** 2 / (w_cal ** 2).sum()

    # 3. applicability domain ---------------------------------------------
    rng = bundle["train_feature_range"]
    n_out = sum(((df[c] < rng[c][0]) | (df[c] > rng[c][1])).astype(int) for c in NUMERIC)

    # 4-5. risk ------------------------------------------------------------
    r_pt, r_lo, r_hi = glof_risk(df, mu), glof_risk(df, lo), glof_risk(df, hi)
    cls = risk_class(r_lo, r_hi, args.risk_threshold, args.moderate_threshold)

    a1 = df.area_2000_km2.to_numpy()
    out = df[["lake_id", "lat", "lon", "valley", "dam_type", "area_2000_km2"]].copy()
    out["log_change_pred"], out["log_change_lo"], out["log_change_hi"] = mu, lo, hi
    out["interval_width_log"] = hi - lo
    out["area_2020_pred_km2"] = a1 * np.exp(mu)
    out["area_2020_lo_km2"], out["area_2020_hi_km2"] = a1 * np.exp(lo), a1 * np.exp(hi)
    out["pct_change_pred"] = 100 * (np.exp(mu) - 1)
    out["pct_change_lo"], out["pct_change_hi"] = 100 * (np.exp(lo) - 1), 100 * (np.exp(hi) - 1)
    out["trend_call"] = np.select([lo > 0, hi < 0], ["growing", "shrinking"], "uncertain")
    out["shift_weight"] = w_new
    out["n_features_outside_training_range"] = n_out
    out["risk_point"], out["risk_lo"], out["risk_hi"] = r_pt, r_lo, r_hi
    out["risk_class"] = cls
    out.to_csv("outputs/analysis_predictions.csv", index=False)

    # high-risk zones: 0.25° cells ranked by expected number of high-risk lakes
    zd = args.zone_deg
    out["zone"] = (np.floor(out.lat / zd).astype(int).astype(str) + "_"
                   + np.floor(out.lon / zd).astype(int).astype(str))
    z = out.groupby("zone").agg(
        n_lakes=("lake_id", "size"),
        n_high_confident=("risk_class", lambda s: (s == "High (confident)").sum()),
        n_high_possible=("risk_class", lambda s: (s == "High (possible)").sum()),
        max_risk_hi=("risk_hi", "max"), mean_risk_hi=("risk_hi", "mean"),
        mean_interval_width=("interval_width_log", "mean"),
        lat_center=("lat", "mean"), lon_center=("lon", "mean"))
    z["zone_score"] = z.n_high_confident + 0.5 * z.n_high_possible + z.max_risk_hi
    z = z[(z.n_high_confident + z.n_high_possible) > 0].sort_values("zone_score", ascending=False)
    z.insert(0, "rank", np.arange(1, len(z) + 1))
    z.round(4).to_csv("outputs/high_risk_zones.csv")

    bbox = meta["analysis_bbox"]
    map_figure(out, bbox, z, "figures/step3_spatial_uncertainty_maps.png")

    # 7. validation against hidden answer key -----------------------------
    key = pd.read_csv("data/analysis_answer_key.csv").set_index("lake_id").loc[df.lake_id]
    y = key["log_area_change"].to_numpy()
    out["risk_true"] = glof_risk(df, y)
    s_true = np.abs(y - mu) / sg
    cov_all = {k: float(np.mean(s_true <= v)) for k, v in q.items()}
    width_all = {k: float(np.mean(2 * v * sg)) for k, v in q.items()}
    inside = (y >= lo) & (y <= hi)
    contact = np.exp(-df.dist_to_glacier_km / 0.8).to_numpy()
    groups = {f"dam: {t}": (df.dam_type == t).to_numpy() for t in ["moraine", "ice", "bedrock"]}
    groups.update({"glacier contact": contact > 0.5, "detached": contact <= 0.5,
                   "≥1 feature outside train range": n_out >= 1, "drained lakes": key.drainage_event.to_numpy() == 1})
    cov_g = {k: float(inside[m].mean()) for k, m in groups.items() if m.sum() >= 10}
    true_high = out.risk_true.to_numpy() >= args.risk_threshold
    flagged = np.char.startswith(cls.astype(str), "High")
    conf = cls == "High (confident)"
    res = {
        "n_lakes": int(len(df)), "alpha": alpha,
        "R2": round(float(r2_score(y, mu)), 4),
        "method_used": args.method,
        "coverage": {k: round(v, 4) for k, v in cov_all.items()},
        "mean_width": {k: round(v, 4) for k, v in width_all.items()},
        "coverage_by_group": {k: round(v, 4) for k, v in cov_g.items()},
        "risk_interval_coverage": round(float(np.mean((out.risk_true >= r_lo - 1e-12) & (out.risk_true <= r_hi + 1e-12))), 4),
        "truly_high_risk_lakes": int(true_high.sum()),
        "recall_of_truly_high_by_High_flags": round(float(flagged[true_high].mean()), 4) if true_high.any() else None,
        "precision_High_confident": round(float(true_high[conf].mean()), 4) if conf.any() else None,
        "recall_point_estimate_only": round(float((r_pt >= args.risk_threshold)[true_high].mean()), 4) if true_high.any() else None,
        "effective_sample_size_of_weights": round(float(ess), 1),
        "lakes_with_feature_outside_training_range": int((n_out >= 1).sum()),
        "risk_class_counts": out.risk_class.value_counts().to_dict(),
        "high_risk_zones": int(len(z)),
    }
    rep = None
    if args.replicates > 0:
        try:
            rep = replicate_check(bundle, args.replicates, meta["analysis_seed"] + 1)
        except ModuleNotFoundError as err:
            print(f"Replicate check skipped (needs Step 1): {err}")
    if rep is not None:
        rep.to_csv("outputs/step3_replicate_coverage.csv", index=False)
        res["replicates"] = {"n": int(len(rep)), "R2_mean": round(float(rep.R2.mean()), 4)}
        for k in METHODS:
            res["replicates"][k] = {
                "coverage_mean": round(float(rep[f"coverage_{k}"].mean()), 4),
                "coverage_min": round(float(rep[f"coverage_{k}"].min()), 4),
                "coverage_max": round(float(rep[f"coverage_{k}"].max()), 4),
                "mean_width": round(float(rep[f"width_{k}"].mean()), 4)}
    json.dump(res, open("outputs/step3_validation.json", "w"), indent=2)
    validation_figure(res, out, alpha, args.risk_threshold, "figures/step3_validation.png")

    print("STEP 3 complete - range B analysed with the frozen range-A model")
    print(f"  R² on range B                    : {res['R2']:.3f}")
    for k in METHODS:
        tag = "  <- used" if k == args.method else ""
        print(f"  Coverage {k:<18s}: {cov_all[k]:.3f}  (mean width {width_all[k]:.3f}){tag}")
    print(f"  Weight ESS                       : {ess:.0f} of {len(w_cal)} calibration lakes")
    for k, v in cov_g.items():
        print(f"    {k:<32s} {v:.3f}  (n={groups[k].sum()})  [{args.method}]")
    if rep is not None:
        r = res["replicates"]
        print(f"  Robustness: {r['n']} more independent range-B datasets (mean R² {r['R2_mean']:.3f})")
        for k in METHODS:
            print(f"    {k:<18s} coverage mean {r[k]['coverage_mean']:.3f}  "
                  f"[{r[k]['coverage_min']:.3f} - {r[k]['coverage_max']:.3f}]  width {r[k]['mean_width']:.3f}")
    print(f"  Risk classes: {res['risk_class_counts']}")
    print(f"  Truly high-risk lakes: {res['truly_high_risk_lakes']} | caught by High flags: "
          f"{res['recall_of_truly_high_by_High_flags']} | by point estimate only: {res['recall_point_estimate_only']}")
    print(f"  Precision of High (confident): {res['precision_High_confident']}")
    print(f"  High-risk zones: {len(z)}; top 5:")
    print(z.head(5)[["rank", "n_lakes", "n_high_confident", "n_high_possible", "max_risk_hi",
                     "lat_center", "lon_center"]].round(3).to_string())


if __name__ == "__main__":
    main()

STEP 3 complete - range B analysed with the frozen range-A model
  R² on range B                    : 0.881
  Coverage standard          : 0.917  (mean width 0.357)
  Coverage mondrian          : 0.929  (mean width 0.397)  <- used
  Coverage mondrian-weighted : 0.903  (mean width 1.108)
  Weight ESS                       : 134 of 916 calibration lakes
    dam: moraine                     0.915  (n=892)  [mondrian]
    dam: ice                         0.949  (n=256)  [mondrian]
    dam: bedrock                     0.952  (n=352)  [mondrian]
    glacier contact                  0.926  (n=673)  [mondrian]
    detached                         0.932  (n=827)  [mondrian]
    ≥1 feature outside train range   0.928  (n=684)  [mondrian]
  Robustness: 20 more independent range-B datasets (mean R² 0.866)
    standard           coverage mean 0.907  [0.895 - 0.919]  width 0.361
    mondrian           coverage mean 0.925  [0.915 - 0.933]  width 0.401
    mondrian-weighted  coverage mean 0.904  [0.88